# 注意力之機制：MHA、MQA、GQA 與 MLA

> **難度：** 中等 | **時長：** 約五十分

注意力機制之選擇，乃 LLM 推理服務中 KV-cache 存量削減之首要樞紐。本篇自多頭注意力（MHA）始，循序而至多潛變量注意力（MLA），逐一展示各變體於存量、品質與計算複雜度間之取捨。

**所習者：**
- MHA 為各頭分別計算 K/V 之法
- MQA 令諸頭共用一 K/V 之法（Falcon）
- GQA 以分組兼顧存量與品質之法（LLaMA-2/3）
- MLA 以低秩投影壓縮 K/V 之法（DeepSeek-V2）
- 依真實模型配置作 KV-cache 定量較量
- 其餘變種：線性注意力、滑動窗口注意力、稀疏注意力
- 諸變體於張量並行（TP）下之異同（切分與複製 KV 頭之別）

**前備知識：** [00-kv-cache](00-kv-cache.ipynb)（明 KV-cache 之義及其要），[02-tensor-parallelism](../../02-tensor-parallelism.ipynb)（第七節所需）

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import sys, os

sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))

from mp_tutorial.inference import (
    calc_kv_cache_size,
    attention_forward_sim,
)
from mp_tutorial.inference_viz import (
    draw_attention_head_layout,
    draw_kv_cache_comparison_bar,
    draw_mla_projection_flow,
    draw_mha_vs_mqa_vs_gqa,
)
from mp_tutorial.formatting import info_box, comparison_table

import warnings
warnings.filterwarnings('ignore')

from mp_tutorial.fonts import configure_cjk_fonts
configure_cjk_fonts()

torch.manual_seed(42)
print('Ready!')

## 一、多頭注意力（MHA）—— 基準

標準多頭注意力（Vaswani et al., 2017），將輸入分投為**各頭獨立之 Q、K、V**。各頭獨立運算注意力，而後合併其果。

$$\text{head}_i = \text{Softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}}\right) V_i$$

$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

自回歸解碼之際，往昔諸 token 之 K 與 V 皆須緩存。KV-cache 每 token 每層存 **2 張量（K 與 V）× n_heads × head_dim** 之值。

In [ ]:
# 小型 MHA 示例：四頭，d_model=16，seq_len=4
batch, seq_len, d_model = 1, 4, 16
n_heads = 4
head_dim = d_model // n_heads  # 4

x = torch.randn(batch, seq_len, d_model)

result = attention_forward_sim("mha", x, n_heads=n_heads, n_kv_heads=n_heads, head_dim=head_dim)

print("=== MHA 張量之形 ===")
print(f"輸入:     {list(x.shape)}           (batch, seq_len, d_model)")
print(f"Q:        {list(result['q'].shape)}        (batch, n_heads, seq_len, head_dim)")
print(f"K:        {list(result['k'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"V:        {list(result['v'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"分數:     {list(result['scores'].shape)}      (batch, n_heads, seq_len, seq_len)")
print(f"輸出:     {list(result['output'].shape)}       (batch, seq_len, n_heads * head_dim)")
print()
print(f"每 token 之 KV cache：2 × {n_heads} 頭 × {head_dim} 維 = {2 * n_heads * head_dim} 值")

In [ ]:
# 圖示 MHA 頭部佈局
draw_attention_head_layout("mha", n_q_heads=4, n_kv_heads=4, head_dim=4)
plt.show()

In [ ]:
# 大規模下 KV cache 之耗：LLaMA-2-70B 配置（設用 MHA）
mha_cache = calc_kv_cache_size(
    variant="mha", n_heads=64, n_kv_heads=64,
    head_dim=128, seq_len=4096, n_layers=80
)
print(f"LLaMA-2-70B（設用 MHA）：4096 token 之 KV-cache = {mha_cache / 1024**3:.2f} GB")
print(f"此乃單一請求耳！三十二請求之批，僅 KV-cache 便需 {32 * mha_cache / 1024**3:.1f} GB。")

**要旨：** MHA 之 KV-cache 按 `2 × n_layers × n_heads × head_dim × seq_len` 而長。頭數眾多之大模型，此項常為推理服務之首要存量瓶頸——長序列時甚或過於模型權重本身。

後述三種變體，皆致力於**削減 KV-cache 所存之量**。

## 二、多查詢注意力（MQA）—— 諸頭共用 K/V

多查詢注意力（Shazeer, 2019）取至簡之削減法：**諸查詢頭共用同一 K 與 V**。

$$Q_i = X W_i^Q \quad (\text{各頭獨立})$$
$$K = X W^K, \quad V = X W^V \quad (\text{諸頭共用})$$

此法將 KV-cache 削減 `n_heads` 倍——自六十四對 K/V 減至僅一對。K/V 投影矩陣亦小，參數量略減。

**用此法之模型：** Falcon-40B、PaLM、StarCoder

In [ ]:
# 小型 MQA 示例：四查詢頭，一 KV 頭
result_mqa = attention_forward_sim("mqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)

print("=== MQA 張量之形 ===")
print(f"Q:        {list(result_mqa['q'].shape)}        (batch, n_heads, seq_len, head_dim)  — 各頭仍獨立")
print(f"K:        {list(result_mqa['k'].shape)}        (batch, 1, seq_len, head_dim)  — 唯一共用 K")
print(f"V:        {list(result_mqa['v'].shape)}        (batch, 1, seq_len, head_dim)  — 唯一共用 V")
print(f"輸出:     {list(result_mqa['output'].shape)}       (batch, seq_len, d_model)")
print()
print(f"每 token 之 KV cache：2 × 1 頭 × {head_dim} 維 = {2 * 1 * head_dim} 值")
print(f"較 MHA 縮減：{n_heads}×！")

In [ ]:
# 圖示：MQA 中廣播之理
draw_attention_head_layout("mqa", n_q_heads=4, n_kv_heads=1, head_dim=4)
plt.show()

In [ ]:
# 大規模 MQA cache 之省：Falcon-40B 配置
mqa_cache = calc_kv_cache_size(
    variant="mqa", n_heads=64, n_kv_heads=1,
    head_dim=128, seq_len=4096, n_layers=60
)
print(f"Falcon-40B (MQA)：4096 token 之 KV-cache = {mqa_cache / 1024**3:.4f} GB")
print(f"同配置下 MHA：{calc_kv_cache_size('mha', 64, 64, 128, 4096, 60) / 1024**3:.2f} GB")
print(f"縮減：{64}×")

**取捨：** MQA 存量之省至巨，然或損模型之質。僅一 K/V 表示，諸頭皆迫於同一鍵值空間運算注意力——失其專精之能。實踐中，從頭訓練之 MQA 模型效果尚可（Falcon、PaLM），然將既有 MHA 模型微調為 MQA 則質量或降。

此即 GQA 之所由生：MHA 與 MQA 之折衷也。

## 三、分組查詢注意力（GQA）—— 折衷之道

分組查詢注意力（Ainslie et al., 2023）將查詢頭分為**若干組**，各組共用一 K/V 頭。設有 `n_kv_heads` 組：

- `n_kv_heads = n_heads` → 等於 **MHA**（各頭皆有獨立 K/V）
- `n_kv_heads = 1` → 等於 **MQA**（諸頭共用一 K/V）
- `1 < n_kv_heads < n_heads` → **GQA**（分組共用）

$$\text{group}(i) = \lfloor i \cdot n_{kv} / n_q \rfloor$$

**用此法之模型：** LLaMA-2-70B（八 KV 頭）、LLaMA-3（八 KV 頭）、Mistral（八 KV 頭）

In [ ]:
# 小型 GQA 示例：四查詢頭，二 KV 頭（每組二）
result_gqa = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=2, head_dim=head_dim)

print("=== GQA 張量之形（四 Q 頭，二 KV 頭）===")
print(f"Q:        {list(result_gqa['q'].shape)}        (batch, n_q_heads, seq_len, head_dim)")
print(f"K:        {list(result_gqa['k'].shape)}        (batch, n_kv_heads, seq_len, head_dim)  — 二 KV 頭")
print(f"V:        {list(result_gqa['v'].shape)}        (batch, n_kv_heads, seq_len, head_dim)")
print(f"輸出:     {list(result_gqa['output'].shape)}       (batch, seq_len, d_model)")
print()
print(f"Q 頭 0,1 共用 KV 頭 0 | Q 頭 2,3 共用 KV 頭 1")
print(f"每 token 之 KV cache：2 × 2 頭 × {head_dim} 維 = {2 * 2 * head_dim} 值")
print(f"較 MHA 縮減：{n_heads // 2}×")

In [ ]:
# 圖示 GQA 分組
draw_attention_head_layout("gqa", n_q_heads=8, n_kv_heads=2, head_dim=128)
plt.show()

In [ ]:
# 驗證：GQA 可退化為 MHA 與 MQA
print("GQA 為一般化之框架：")
print()

# n_kv_heads = n_heads 時 -> MHA
result_gqa_as_mha = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=4, head_dim=head_dim)
result_pure_mha = attention_forward_sim("mha", x, n_heads=4, n_kv_heads=4, head_dim=head_dim)
match_mha = torch.allclose(result_gqa_as_mha['output'], result_pure_mha['output'], atol=1e-6)
print(f"GQA(n_kv=n_q=4) == MHA: {match_mha}")

# n_kv_heads = 1 時 -> MQA
result_gqa_as_mqa = attention_forward_sim("gqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)
result_pure_mqa = attention_forward_sim("mqa", x, n_heads=4, n_kv_heads=1, head_dim=head_dim)
match_mqa = torch.allclose(result_gqa_as_mqa['output'], result_pure_mqa['output'], atol=1e-6)
print(f"GQA(n_kv=1)    == MQA: {match_mqa}")

In [ ]:
# 以既有圖示函數概覽三法
draw_mha_vs_mqa_vs_gqa(n_q_heads=8, head_dim=128)
plt.show()

**要旨：** GQA 於存量與品質間提供可調之機。LLaMA-2-70B 用六十四查詢頭而僅八 KV 頭——KV-cache 削減八倍而保 MHA 品質之大半。此已為大模型之業界通行配置。

然能更進一步乎？若非削 KV 頭之*數*，而壓其*表示*本身，何如？

## 四、多潛變量注意力（MLA）—— 低秩 KV 壓縮

多潛變量注意力（DeepSeek-AI, 2024）取截然不同之路。MLA 非令查詢組共用 KV 頭，而於緩存前將 KV 表示**壓縮**入低秩潛空間。

其要：
1. **下投影**：壓輸入為潛變量：$c = X W_{\text{down}}$，其中 $c \in \mathbb{R}^{d_{\text{compressed}}}$
2. **緩存**：僅存壓縮後之潛變量 $c$（遠小於分立之 K、V）
3. **上投影**：計算注意力時還原為 K 與 V：$K = c W_{\text{up}}^K$，$V = c W_{\text{up}}^V$

緩存中每 token 僅存一 $d_{\text{compressed}}$ 維向量，而非分別存 $n_{\text{heads}} \times d_{\text{head}}$ 維之 K 與 V。

**用此法之模型：** DeepSeek-V2、DeepSeek-V3

In [ ]:
# 小型 MLA 示例：將 d_model=16 壓縮至 d_compressed=6
d_compressed = 6

result_mla = attention_forward_sim(
    "mla", x, n_heads=4, n_kv_heads=4, head_dim=head_dim, d_compressed=d_compressed
)

print("=== MLA 張量之形 ===")
print(f"Q:              {list(result_mla['q'].shape)}        (batch, n_heads, seq_len, head_dim)")
print(f"壓縮後之 KV:    {list(result_mla['compressed_kv'].shape)}       (batch, seq_len, d_compressed) ← 此為所緩存者")
print(f"K（解壓後）:    {list(result_mla['k'].shape)}        (batch, n_heads, seq_len, head_dim) — 重建之")
print(f"V（解壓後）:    {list(result_mla['v'].shape)}        (batch, n_heads, seq_len, head_dim) — 重建之")
print(f"輸出:           {list(result_mla['output'].shape)}       (batch, seq_len, d_model)")
print()
mha_cache_per_token = 2 * n_heads * head_dim
mla_cache_per_token = d_compressed
print(f"MHA 所緩存：2 × {n_heads} × {head_dim} = {mha_cache_per_token} 值/token")
print(f"MLA 所緩存：{d_compressed} 值/token（壓縮潛變量）")
print(f"壓縮比：{mha_cache_per_token / mla_cache_per_token:.1f}×")

In [ ]:
# 圖示 MLA 流水線
draw_mla_projection_flow(d_model=512, d_compressed=64, n_heads=8, head_dim=64)
plt.show()

In [ ]:
# 大規模 MLA cache 之省：DeepSeek-V2 配置
# DeepSeek-V2：128 頭，head_dim=128，60 層，d_compressed=512
mla_cache = calc_kv_cache_size(
    variant="mla", n_heads=128, n_kv_heads=128,
    head_dim=128, seq_len=4096, n_layers=60, d_compressed=512
)
mha_equivalent = calc_kv_cache_size(
    variant="mha", n_heads=128, n_kv_heads=128,
    head_dim=128, seq_len=4096, n_layers=60
)
print(f"DeepSeek-V2 (MLA, d_c=512)：KV-cache = {mla_cache / 1024**3:.4f} GB")
print(f"同模型用 MHA：              KV-cache = {mha_equivalent / 1024**3:.2f} GB")
print(f"縮減：{mha_equivalent / mla_cache:.0f}×")

**取捨：** MLA 壓縮之極，然計算之耗亦增——自壓縮潛變量至 K/V 之上投影，每次注意力運算皆須行之。此乃以算換存之策：增矩陣乘法而大幅縮 KV-cache。於存量受限之推理負載（長序列、大批次），此交易甚為有利。

## 五、綜合較量

以真實模型配置，將四種變體並列較量。

In [ ]:
# 以真實模型配置並排比較 KV cache
configs = [
    {"name": "MHA\n(假設 70B)", "variant": "mha",
     "n_heads": 64, "n_kv_heads": 64, "head_dim": 128, "n_layers": 80},
    {"name": "MQA\n(Falcon-40B)", "variant": "mqa",
     "n_heads": 64, "n_kv_heads": 1, "head_dim": 128, "n_layers": 60},
    {"name": "GQA\n(LLaMA-2-70B)", "variant": "gqa",
     "n_heads": 64, "n_kv_heads": 8, "head_dim": 128, "n_layers": 80},
    {"name": "MLA\n(DeepSeek-V2)", "variant": "mla",
     "n_heads": 128, "n_kv_heads": 128, "head_dim": 128, "n_layers": 60,
     "d_compressed": 512},
]

draw_kv_cache_comparison_bar(configs)
plt.show()

In [ ]:
# 總結表
print("┌──────────┬──────────────┬───────────────┬──────────────────────────────────────┐")
print("│ 變體     │ KV 頭數      │ 緩存/Token    │ 核心取捨                             │")
print("├──────────┼──────────────┼───────────────┼──────────────────────────────────────┤")
print("│ MHA      │ n_heads      │ 2·h·d         │ 品質完備，存量之耗亦全               │")
print("│ MQA      │ 1            │ 2·d           │ 省至極，品質或損                     │")
print("│ GQA      │ g（可調）    │ 2·g·d         │ 折衷之道——業界通行                   │")
print("│ MLA      │ 無（潛變量） │ d_compressed  │ 壓縮至極，計算之耗增                 │")
print("└──────────┴──────────────┴───────────────┴──────────────────────────────────────┘")
print()
print("h = n_heads, d = head_dim, g = n_kv_groups")
print()
print("演進：MHA (2017) → MQA (2019) → GQA (2023) → MLA (2024)")
print("每步皆以某取捨削 KV-cache 之存量。")

## 六、注意力之其餘變種：超越 Softmax

MHA→MQA→GQA→MLA 一族皆用標準 softmax 注意力——所減者為**緩存之物**，非注意力之算法。另有一路研究，直攻注意力 O(n²) 之計算複雜度。

### 線性注意力（Linear Attention）

標準注意力算 $\text{Softmax}(QK^T)V$，須實例化 $n \times n$ 之注意力矩陣。線性注意力（Katharopoulos et al., 2020）以核函數 $\phi$ 代 softmax：

$$\text{標準：} \quad O = \text{Softmax}(QK^T) \cdot V \qquad \mathcal{O}(n^2 d)$$
$$\text{線性：} \quad O = \phi(Q) \cdot (\phi(K)^T V) \qquad \mathcal{O}(n d^2)$$

其巧在**結合律**：先算 $\phi(K)^T V$（一 $d \times d$ 矩陣），遂全避 $n \times n$ 之積。注意力於序列長度遂為 **O(n)**，然常數因子 $d^2$ 使其僅於甚長序列方顯優勢。

**後繼者：** RetNet（多尺度保留）、RWKV（線性遞推為注意力）、Mamba/S4（狀態空間模型——嚴格言之非「注意力」，然解同一問題）。此等架構能以 O(n) 處理序列，維**固定大小之狀態**而非日增之 KV-cache。

### 滑動窗口注意力（Sliding Window Attention）

滑動窗口注意力（Beltagy et al., 2020；用於 Mistral、Gemma-2）限各 token 僅於固定窗口 $w$ 內運算注意力：

$$\text{Attn}(q_i) = \text{Softmax}(q_i \cdot K[i-w:i]^T) \cdot V[i-w:i]$$

此為 KV-cache 設**硬上界**：不論序列總長幾何，緩存至多 $w$ token。窗外之資訊經層疊間接傳播：$L$ 層 × 窗口 $w$ = 有效感受野 $L \times w$ token。

**用此法之模型：** Mistral-7B（$w = 4096$）、Gemma-2（全局層與滑動窗口層交替）

### 稀疏注意力（Sparse Attention）

Longformer 與 BigBird 合**局部窗口**（各 token 觀鄰近）與**全局 token**（少數 token 觀全部）。得 O(n) 之複雜度而保長距離依賴。

**要旨：** 此等方法與 MQA/GQA/MLA **大抵正交**。可以滑動窗口配 GQA（Mistral 正如此），亦可配 MLA。所減者各異：頭共用減每 token 之 KV-cache *大小*，稀疏/線性法減參與注意力之 *token 數量*。

## 七、注意力變體與張量並行

大模型多卡推理之際，張量並行（TP）將各層切分於眾 GPU。注意力層之標準法（Megatron-LM）為 **Q/K/V 列並行投影**（依頭切分），繼以**輸出投影行並行**與一次 AllReduce。然諸注意力變體與 TP 之交互，迥然各異。

### MHA + TP：至淨之態

`n_heads` 個獨立 Q、K、V 頭，TP 切分直截了當：
- 切分：各 GPU 得 `n_heads / tp_size` 完整之頭（Q、K、V 各一份）
- 計算：各 GPU 獨立行注意力——**零通信**
- 合併：行並行 $W_O$ 後一次 AllReduce

六十四頭分於八卡 → 每卡八頭，整除，完美。

### MQA + TP：複製之困

MQA 僅 **1 KV 頭**而有 `n_heads` Q 頭。一頭不可切：

1. **每卡複製 KV**（通行之法）：各 GPU 持完整之單一 K/V 頭，加自身所分之 Q 頭。僅一頭耳，複製之耗微不足道。
2. **自一卡廣播 KV**：rank 0 算 KV 而後廣播。多一通信步驟，然免冗餘計算。

實踐取方案一，蓋單一 KV 頭至小，複製之費猶廉於廣播之延。

### GQA + TP：視比例而定

關鍵所在：**`n_kv_heads` 能為 `tp_size` 整除否？**

| 場景 | 例 | 策略 |
|------|------|------|
| `n_kv_heads >= tp_size` 且整除 | 八 KV 頭 on 八 GPU | 切分：每卡一 KV 頭 |
| `n_kv_heads >= tp_size` 然不整除 | 八 KV 頭 on 六 GPU | 不可均分——宜避此配 |
| `n_kv_heads < tp_size` | 八 KV 頭 on 十六 GPU | **複製**：各 KV 頭複製於 `tp_size / n_kv_heads` 卡 |

LLaMA-2-70B 之八 KV 頭於八卡乃甜點之配：每卡恰得一 KV 頭，八 Q 頭在本地共用之。

**實用之諫：** 設計新模型時，`n_kv_heads` 當取能整除目標 `tp_size` 之值。八 KV 頭之所以常見，正因其能整除八、十六、三十二、六十四卡。

### MLA + TP：切分上投影

MLA 之 TP 策略根本殊異，蓋無 KV「頭」可切：

1. **下投影** $W_{\text{down}}$：產共享之壓縮潛變量 $c$。須完整計算（或複製），蓋諸頭皆需之。
2. **壓縮潛變量**：於諸 GPU 上複製（至小：每 token 僅 `d_compressed`）。
3. **上投影** $W_{\text{up}}^K$、$W_{\text{up}}^V$：列並行——各 GPU 上投影至自身所負之頭。
4. **注意力 + 輸出投影**：自此與 MHA 同——各卡有其局部之頭。

壓縮潛變量之複製正為 MLA 於 TP 下之利：無須複製 `2 × n_heads × head_dim` 每 token（MHA），僅須複製 `d_compressed`——省 cache 之同一壓縮機制亦減 TP 通信量。

In [ ]:
# 諸變體之 TP 策略一覽
print("┌──────────┬────────────────────┬──────────────────────┬─────────────────────────────┐")
print("│ 變體     │ Q 切分             │ KV 切分              │ TP 所宜注意者               │")
print("├──────────┼────────────────────┼──────────────────────┼─────────────────────────────┤")
print("│ MHA      │ n_heads / tp_size  │ n_heads / tp_size    │ 無——至淨之切分              │")
print("│ MQA      │ n_heads / tp_size  │ 複製（一頭）         │ KV 複製而非切分             │")
print("│ GQA      │ n_heads / tp_size  │ n_kv / tp_size *     │ * n_kv < tp 時須複製        │")
print("│ MLA      │ n_heads / tp_size  │ 複製潛變量           │ 上投影為列並行              │")
print("└──────────┴────────────────────┴──────────────────────┴─────────────────────────────┘")
print()

# 具體之例：LLaMA-2-70B GQA 於不同 TP 規模下之狀
n_q, n_kv = 64, 8
for tp in [1, 2, 4, 8, 16]:
    q_per_gpu = n_q // tp
    if n_kv >= tp:
        kv_per_gpu = n_kv // tp
        kv_strategy = f"每卡 {kv_per_gpu} KV 頭（切分）"
    else:
        replicas = tp // n_kv
        kv_strategy = f"每 KV 頭複製於 {replicas} 卡（複製）"
    print(f"  TP={tp:2d}: 每卡 {q_per_gpu:2d} Q 頭, {kv_strategy}")

## 總結

| 變體 | 論文 | 緩存公式 | 真實案例 | 相對緩存量 |
|------|------|----------|----------|------------|
| **MHA** | Vaswani 2017 | `2 · n_heads · d_head` / token / 層 | — | 1×（基準） |
| **MQA** | Shazeer 2019 | `2 · d_head` / token / 層 | Falcon-40B, PaLM | 1/n_heads × |
| **GQA** | Ainslie 2023 | `2 · n_kv_heads · d_head` / token / 層 | LLaMA-2-70B (8 KV) | 1/group_size × |
| **MLA** | DeepSeek 2024 | `d_compressed` / token / 層 | DeepSeek-V2 (512) | d_c / (2·h·d) × |

**要旨：**
- KV-cache 存量乃大規模 LLM 推理之瓶頸
- GQA 為當今業界之通行（LLaMA、Mistral、Gemma）——簡明、有效、可調
- MLA 以低秩投影更進一步壓縮，代價為額外計算
- 抉擇視部署約束而定：存量受限 → 宜 MLA/MQA，算力受限 → MHA/GQA 即足

**延伸閱讀：**
- [00-kv-cache](00-kv-cache.ipynb) — KV-cache 之義及其要
- [01-flash-attention](01-flash-attention.ipynb) — 高效注意力之*計算*（與頭共用正交）
- [03-paged-attention](03-paged-attention.ipynb) — 高效注意力之*存量管理*
- [08-serving-architecture](08-serving-architecture.ipynb) — 此等選擇如何影響推理系統設計